In [2]:
import pandas as pd

# Load the CSV
df = pd.read_csv("data/data_hour.csv")

# Print column names to verify structure
print("Columns before:", df.columns)

# Drop the first column by position
df = df.drop(df.columns[0], axis=1)

# Print column names to verify deletion
print("Columns after:", df.columns)

# Save back to the same file with index (optional, see below)
df.to_csv("data/data_hour.csv", index=False)  # Often you do NOT want index=True

Columns before: Index(['Unnamed: 0', 'Datetime', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')
Columns after: Index(['Datetime', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')


In [13]:
import pandas as pd

# ---------- Load and process df1 ----------
df1 = pd.read_csv("data/data.csv", parse_dates=["Datetime"])
df1["Datetime"] = pd.to_datetime(df1["Datetime"], utc=True)
df1.set_index("Datetime", inplace=True)
df1.index = df1.index.tz_convert("America/Chicago")

# ---------- Load and process df2 ----------
df2 = pd.read_csv("data/data_2.csv", parse_dates=["Datetime"])
df2["Datetime"] = pd.to_datetime(df2["Datetime"], utc=True)
df2.set_index("Datetime", inplace=True)
df2.index = df2.index.tz_convert("America/Chicago")

# ---------- Merge ----------
merged_df = pd.merge(df1, df2, how="outer", left_index=True, right_index=True)

merged_sorted = merged_df.sort_index()
merged_df.reset_index(inplace=True)
merged_df_after_reset = merged_df.set_index("Datetime")


In [14]:
# Iterate through all columns ending with _x
for col in merged_df.columns:
    if col.endswith("_x"):
        base_col = col[:-2]  # Remove "_x"
        col_x = col
        col_y = f"{base_col}_y"

        if col_y in merged_df.columns:
            # Combine _x and _y with _x having priority unless it's NaN
            merged_df[base_col] = merged_df[col_x].combine_first(merged_df[col_y])

            # Drop the original _x and _y columns
            merged_df.drop(columns=[col_x, col_y], inplace=True)


# ---------- Combine duplicate _x/_y columns ----------
for col in merged_df.columns:
    if col.endswith("_x"):
        base_col = col[:-2]
        col_x = col
        col_y = f"{base_col}_y"

        if col_y in merged_df.columns:
            merged_df[base_col] = merged_df[col_x].combine_first(merged_df[col_y])
            merged_df.drop(columns=[col_x, col_y], inplace=True)

# ---------- Save ----------
merged_df.to_csv("data/merged_data.csv", index=False)